In [ ]:
%load_ext autoreload
%autoreload 2

# core_v2

> updated variant of the motif-finding algorithm in core.py. Both versions ship in this package; core.py is left untouched, and callers choose which one they want by importing from the corresponding module (randseq.core vs randseq.core_v2) rather than a flag or config option.

In [ ]:
#| default_exp core_v2

In [ ]:
#| hide
from nbdev.showdoc import *

This module overrides four functions from core.py: find_restricted_motifs,
find_restricted_motifs_mp, filter_redundant_patterns and
update_motif_scores_from_unique_hits. The main behavioural change is an
updated filter_redundant_patterns that is symmetric:

  * Drops a SPECIFIC motif when the GENERAL one is not significantly worse
    (existing behaviour, score2 > score1 - margin).
  * Drops a GENERAL motif when the SPECIFIC one is significantly better
    (new behaviour, score1 > score2 + margin).

Everything else this module calls (identify_depleted_motifs_scanning_ends,
filter_to_core_motifs, filter_sequences_without_core_motifs, get_sites_in_seq,
get_fold_change_values_per_site, get_sites_scores, score,
process_single_flexible_pattern) is reused unchanged from core.py and
imported explicitly by name below, rather than via `from .core import *`,
so it's clear at a glance what's new here versus what's shared.

In [ ]:
#| export
import functools
import pandas as pd
from multiprocessing import Pool
from tqdm import tqdm
import warnings

import numpy as np

from randseq.core import (identify_depleted_motifs_scanning_ends, filter_to_core_motifs, filter_sequences_without_core_motifs, get_sites_in_seq, get_fold_change_values_per_site, get_sites_scores, score, process_single_flexible_pattern)
from randseq.utils import find_broad_in_specific, get_lib_seq_context, _get_filter_for_motif  # needed by updated functions

In [ ]:
#| export
def update_motif_scores_from_unique_hits(
    flexible_motifs_df,
    log2fc_series,
    left_context,
    right_context,
    flexible_motif_log2fc_thr=-1,
    flexible_motif_min_support=3,
    flexible_motif_score_thr=0.85
):
    """
    Overrides core.py version to add tqdm progress feedback.

    Updates a DataFrame of motifs with new scores based on unique sequence
    hits (sequences not matched by any other motif) and filters the result.
    """
    new_metrics = []
    all_motifs = flexible_motifs_df['motif'].tolist()

    # Pre-compute filter for every motif once to avoid redundant .apply() calls.
    motif_filters = {}
    print("  Pre-computing sequence filters for each motif...")
    for m in tqdm(all_motifs, desc="  Building filters"):
        motif_filters[m] = _get_filter_for_motif(log2fc_series, m, left_context, right_context)

    print("  Scoring motifs on unique hits...")
    for current_motif in tqdm(all_motifs, desc="  Scoring motifs"):
        matches_current_filter = motif_filters[current_motif]

        matches_other_filter = pd.Series(False, index=log2fc_series.index)
        for other_motif, other_filter in motif_filters.items():
            if other_motif != current_motif:
                matches_other_filter = matches_other_filter | other_filter

        final_filter = matches_current_filter & ~matches_other_filter
        filtered_series = log2fc_series[final_filter]

        log2fc_values = filtered_series.values
        num_sequences = len(log2fc_values)
        fraction_depleted = score(log2fc_values, thr=flexible_motif_log2fc_thr)
        avg_log2fc = np.mean(log2fc_values) if num_sequences > 0 else 0.0

        new_metrics.append({
            'fraction_depleted': fraction_depleted,
            'num_sequences': num_sequences,
            'avg_log2fc': avg_log2fc
        })

    df_updated = flexible_motifs_df.copy()
    metrics_df = pd.DataFrame(new_metrics, index=df_updated.index)

    df_updated['fraction_depleted'] = metrics_df['fraction_depleted']
    df_updated['num_sequences'] = metrics_df['num_sequences']
    df_updated['avg_log2fc'] = metrics_df['avg_log2fc']

    final_df = df_updated[
        (df_updated['avg_log2fc'] <= flexible_motif_log2fc_thr) &
        (df_updated['num_sequences'] >= flexible_motif_min_support) &
        (df_updated['fraction_depleted'] >= flexible_motif_score_thr)
    ]

    return final_df

In [ ]:
#| export
def filter_redundant_patterns(df, score_margin=0.05):
    """
    Filters a DataFrame of patterns, removing redundant motifs based on
    specificity and a score condition.

    A motif 'p1' is considered redundant and removed if there is another
    motif 'p2' in the DataFrame such that:

    1. p1 is a specific version of p2 AND
       p2.score > p1.score - score_margin
       (general is not significantly worse  → drop specific)

    OR

    2. p1 is a general version of p2 AND
       p2.score > p1.score + score_margin
       (specific is significantly better → drop general)

    3. For equivalent motifs the one with the lower score is dropped.
       If scores are equal, the one with the higher reset-index is dropped.

    Args:
        df (pd.DataFrame): DataFrame with 'motif' and 'fraction_depleted'
            columns.
        score_margin (float): Margin used to relax score comparisons.

    Returns:
        pd.DataFrame: A filtered DataFrame with redundant rows removed.
    """
    indices_to_drop = set()
    df_reset = df.reset_index()

    for i, p1 in df_reset.iterrows():
        if i in indices_to_drop:
            continue

        for j, p2 in df_reset.iterrows():
            if i == j or j in indices_to_drop:
                continue

            motif1, score1 = p1['motif'], p1['fraction_depleted']
            motif2, score2 = p2['motif'], p2['fraction_depleted']

            is_p1_subset_of_p2 = find_broad_in_specific(motif1, motif2)

            if not is_p1_subset_of_p2:
                continue

            is_p2_subset_of_p1 = find_broad_in_specific(motif2, motif1)

            if not is_p2_subset_of_p1:
                # p1 is STRICTLY more specific than p2 (p2 is the general motif).

                if score2 > score1 - score_margin:
                    # General motif (p2) is not significantly worse → drop
                    # the specific one (p1).
                    indices_to_drop.add(i)
                    break

                elif score1 > score2 + score_margin:
                    # Specific motif (p1) is significantly better → drop
                    # the general one (p2).
                    indices_to_drop.add(j)
                    # Do NOT break: p1 might still be redundant w.r.t. other
                    # motifs in a later iteration.

            else:
                # p1 and p2 are equivalent.  Drop the worse one; break ties
                # by higher reset-index.
                if score1 < score2:
                    indices_to_drop.add(i)
                    break
                elif score1 == score2 and i > j:
                    indices_to_drop.add(i)
                    break

    original_indices_to_drop = df_reset.loc[list(indices_to_drop)]['index']
    return df.drop(original_indices_to_drop)

In [ ]:
#| export
def find_restricted_motifs(log2fc_series,
                           left_context_str,
                           right_context_str,
                           fixed_motif_scan_depth_k=6,
                           fixed_motif_max_length=4,
                           fixed_motif_depletion_thr=-1,
                           fixed_motif_score_thr=0.85,
                           fixed_motif_min_support=5,
                           core_motif_score_margin=0.05,
                           flexible_motif_patterns=[(6,0,0)],
                           flexible_motif_log2fc_thr=-1,
                           flexible_motif_min_support=3,
                           flexible_motif_score_thr=0.7,
                           core_flexible_motif_score_margin=0.2,
                           max_candidates=150
                           ):
    """
    Full pipeline to identify fixed-position core motifs and then
    position-independent motifs in the remaining sequences.

    Fixes vs. core.py:
      1. Uses the symmetric filter_redundant_patterns (defined above) during
         the iterative loop, so overly general motifs are also dropped.
      2. Passes log2fc_series_for_flexible_analysis (sequences without
         fixed-position core motifs) to update_motif_scores_from_unique_hits
         instead of the full log2fc_series, keeping the collapsing step
         consistent with the motif-discovery step.

    Args:
        max_candidates (int): Safety limit on the number of candidate flexible
            motifs before entering the O(N²) update_motif_scores_from_unique_hits
            step.  If more candidates are found the function aborts that step,
            prints guidance on tightening thresholds, and returns the raw
            (unfiltered) candidates so the caller can inspect them.
            Default: 150.
    """
    print("--- Starting Fixed-Position Motif Analysis (on original short sequences) ---")
    depleted_fixed_motifs_df = identify_depleted_motifs_scanning_ends(
        log2fc_series,
        scan_depth_k=fixed_motif_scan_depth_k,
        max_motif_length=fixed_motif_max_length,
        depletion_threshold=fixed_motif_depletion_thr,
        score_thr=fixed_motif_score_thr,
        min_sequence_support=fixed_motif_min_support
    )
    core_fixed_motifs_df = filter_to_core_motifs(
        depleted_fixed_motifs_df,
        score_improvement_margin=core_motif_score_margin
    )
    del depleted_fixed_motifs_df

    if core_fixed_motifs_df.empty:
        print("No core motifs found at fixed positions.")
    else:
        print(f"Found {len(core_fixed_motifs_df)} core motifs at fixed positions:")
        for _, row in core_fixed_motifs_df.iterrows():
            print(f"  - Fixed Motif: {row['motif']}, Pos: {row['position']}, Len: {row['length']}, FracDep: {row['fraction_depleted']:.2f}, AvgFC: {row['avg_log2fc']:.2f}, N: {row['num_sequences']}")

    print("\n--- Starting Position-Independent Motif Analysis on Filtered Sequences (with context) ---")
    log2fc_series_for_flexible_analysis = filter_sequences_without_core_motifs(
        log2fc_series,
        core_fixed_motifs_df
    )
    if len(log2fc_series_for_flexible_analysis) == len(log2fc_series):
        print("No sequences were filtered out based on fixed-position core motifs (or no core motifs found).")
    else:
        print(f"Filtered log2fc_series for flexible analysis: {len(log2fc_series_for_flexible_analysis)} sequences remaining.")

    if log2fc_series_for_flexible_analysis.empty:
        print("No sequences remaining for flexible motif analysis after filtering.")
        flexible_motifs_results_df = pd.DataFrame(columns=['motif', 'pattern', 'fraction_depleted', 'num_sequences', 'avg_log2fc'])
        return core_fixed_motifs_df, flexible_motifs_results_df

    short_sequences_for_flex_analysis = log2fc_series_for_flexible_analysis.index.tolist()
    sequences_with_context_for_flex = get_lib_seq_context(
        short_sequences_for_flex_analysis,
        left_context_str,
        right_context_str
    )
    fc_values_for_flexible_analysis = log2fc_series_for_flexible_analysis.tolist()
    # NOTE: log2fc_series_for_flexible_analysis is kept alive — it is passed to
    # update_motif_scores_from_unique_hits below so the collapsing step operates
    # on the same sequence universe used for motif discovery.

    cumulative_flexible_motifs_df = pd.DataFrame()

    if not isinstance(flexible_motif_patterns, list):
        print("Warning: flexible_motif_patterns should be a list. Wrapping it in a list.")
        flexible_motif_patterns = [flexible_motif_patterns]

    flexible_motif_patterns = sorted(flexible_motif_patterns, key=lambda x: (x[0]+x[2], x[1]))

    for current_flex_pattern in tqdm(flexible_motif_patterns):

        site_sets_flexible = get_sites_in_seq(
            sequences_with_context_for_flex,
            pattern=current_flex_pattern,
            no_ori=True
        )
        if len(site_sets_flexible) != len(fc_values_for_flexible_analysis):
            print(f"    Error: Mismatch in length between site sets ({len(site_sets_flexible)}) and FC values ({len(fc_values_for_flexible_analysis)}) for pattern {current_flex_pattern}. Skipping this pattern.")
            continue

        site_fcs_flexible = get_fold_change_values_per_site(
            site_sets_flexible,
            fc_values_for_flexible_analysis
        )
        del site_sets_flexible

        current_pattern_scores_df = get_sites_scores(
            site_fcs_flexible,
            pattern=current_flex_pattern,
            log2FC_thr=flexible_motif_log2fc_thr
        )

        if not current_pattern_scores_df.empty:
            current_pattern_scores_df.rename(columns={'site': 'motif'}, inplace=True)
            current_pattern_scores_df['pattern'] = str(current_flex_pattern)
            filtered_for_this_pattern = current_pattern_scores_df[
                (current_pattern_scores_df['fraction_depleted'] > flexible_motif_score_thr) &
                (current_pattern_scores_df['num_sequences'] > flexible_motif_min_support)
            ]
            if len(filtered_for_this_pattern) > 0:
                print(f"Found hits with pattern: {current_flex_pattern}")
                print(filtered_for_this_pattern)

            if not filtered_for_this_pattern.empty:
                print(f"Found {len(filtered_for_this_pattern)} candidate(s) for pattern {current_flex_pattern}. Filtering against previous results...")

                combined_before_filtering = pd.concat([cumulative_flexible_motifs_df, filtered_for_this_pattern], ignore_index=True)

                # Uses the symmetric filter_redundant_patterns defined above,
                # not the one-sided version from core.py.
                cumulative_flexible_motifs_df = filter_redundant_patterns(
                    combined_before_filtering,
                    score_margin=core_flexible_motif_score_margin,
                )
                print(f"  -> {len(cumulative_flexible_motifs_df)} core motifs remain after filtering.")
                print(cumulative_flexible_motifs_df)

    if cumulative_flexible_motifs_df.empty:
        print(f"\nNo flexible motifs found meeting initial support > {flexible_motif_min_support} and score > {flexible_motif_score_thr} for any provided pattern.")
        flexible_motifs_results_df = pd.DataFrame(columns=['motif', 'pattern', 'fraction_depleted', 'num_sequences', 'avg_log2fc'])
    else:
        n_candidates = len(cumulative_flexible_motifs_df)
        print(f"\nIdentified {n_candidates} raw flexible motifs. Now filtering to core flexible motifs...")

        if n_candidates > max_candidates:
            warnings.warn(
                f"\n{'='*70}\n"
                f"TOO MANY CANDIDATES ({n_candidates} > max_candidates={max_candidates}).\n"
                f"The unique-hit scoring step is O(N²) per sequence and would run\n"
                f"effectively forever with this many candidates.\n\n"
                f"This usually means the thresholds are too permissive for this data.\n"
                f"Suggestions to reduce candidates:\n"
                f"  - Increase flexible_motif_log2fc_thr  (currently {flexible_motif_log2fc_thr}, try e.g. -0.8 or -1.0)\n"
                f"  - Increase flexible_motif_score_thr   (currently {flexible_motif_score_thr}, try e.g. 0.7 or 0.85)\n"
                f"  - Increase flexible_motif_min_support (currently {flexible_motif_min_support}, try e.g. 5 or 10)\n"
                f"  - Increase max_candidates if you know what you are doing.\n"
                f"\nReturning the raw (unfiltered) candidates so you can inspect them.\n"
                f"{'='*70}",
                stacklevel=2
            )
            print(f"\nTop candidates by fraction_depleted:")
            print(cumulative_flexible_motifs_df.sort_values('fraction_depleted', ascending=False).head(20).to_string())
            return core_fixed_motifs_df, cumulative_flexible_motifs_df

        print("\nNow filtering to remove sequences with multiple motifs from the analysis and dropping motifs that no longer pass the thresholds")
        print(f"(Scoring {n_candidates} candidate motifs — this step is O(N²), may take a moment...)")

        # FIX: pass log2fc_series_for_flexible_analysis instead of log2fc_series
        # so sequences already explained by fixed-position motifs are excluded,
        # consistent with how the flexible motifs were originally discovered.
        flexible_motifs_results_df = update_motif_scores_from_unique_hits(
            cumulative_flexible_motifs_df,
            log2fc_series_for_flexible_analysis,
            left_context_str,
            right_context_str,
            flexible_motif_log2fc_thr=flexible_motif_log2fc_thr,
            flexible_motif_min_support=flexible_motif_min_support,
            flexible_motif_score_thr=flexible_motif_score_thr
        )

        print(f"\n{len(flexible_motifs_results_df)} motifs remaining:")
        if flexible_motifs_results_df.empty:
            print("No core flexible motifs remained after filtering.")
        else:
            for _, row in flexible_motifs_results_df.iterrows():
                print(f"{row['motif']} (from pattern {row['pattern']}), FracDep: {row['fraction_depleted']:.2f}, AvgFC: {row['avg_log2fc']:.2f}, N: {row['num_sequences']}")

    return core_fixed_motifs_df, flexible_motifs_results_df

In [ ]:
#| export
def find_restricted_motifs_mp(log2fc_series,
                               left_context_str,
                               right_context_str,
                               fixed_motif_scan_depth_k=6,
                               fixed_motif_max_length=4,
                               fixed_motif_depletion_thr=-1,
                               fixed_motif_score_thr=0.85,
                               fixed_motif_min_support=5,
                               core_motif_score_margin=0.05,
                               flexible_motif_patterns=[(6, 0, 0), (3, 4, 4)],
                               flexible_motif_log2fc_thr=-1,
                               flexible_motif_min_support=3,
                               flexible_motif_score_thr=0.7,
                               core_flexible_motif_score_margin=0.2,
                               cpu_count=1,
                               max_candidates=150):
    """
    Multiprocessing version of find_restricted_motifs with the same fixes as
    core_v2.py's find_restricted_motifs:

      1. Keeps log2fc_series_for_flexible_analysis alive and passes it (not
         the original log2fc_series) to update_motif_scores_from_unique_hits.
      2. max_candidates guard: aborts the O(N²) scoring step and warns the
         user when too many candidates accumulate (noisy/permissive thresholds).
      3. Uses the overridden update_motif_scores_from_unique_hits with tqdm
         progress and pre-computed filters.
      4. Uses the symmetric filter_redundant_patterns from this module.

    Args:
        max_candidates (int): If more candidates than this remain after
            filter_redundant_patterns, abort the scoring step and return the
            raw candidates with threshold-tightening suggestions. Default: 150.
    """
    print("--- Starting Fixed-Position Motif Analysis (on original short sequences) ---")
    depleted_fixed_motifs_df = identify_depleted_motifs_scanning_ends(
        log2fc_series,
        scan_depth_k=fixed_motif_scan_depth_k,
        max_motif_length=fixed_motif_max_length,
        depletion_threshold=fixed_motif_depletion_thr,
        score_thr=fixed_motif_score_thr,
        min_sequence_support=fixed_motif_min_support
    )
    core_fixed_motifs_df = filter_to_core_motifs(
        depleted_fixed_motifs_df,
        score_improvement_margin=core_motif_score_margin
    )
    del depleted_fixed_motifs_df

    if core_fixed_motifs_df.empty:
        print("No core motifs found at fixed positions.")
    else:
        print(f"Found {len(core_fixed_motifs_df)} core motifs at fixed positions:")
        for _, row in core_fixed_motifs_df.iterrows():
            print(f"  - Fixed Motif: {row['motif']}, Pos: {row['position']}, Len: {row['length']}, FracDep: {row['fraction_depleted']:.2f}, AvgFC: {row['avg_log2fc']:.2f}, N: {row['num_sequences']}")

    print("\n--- Starting Position-Independent Motif Analysis on Filtered Sequences (with context) ---")
    log2fc_series_for_flexible_analysis = filter_sequences_without_core_motifs(
        log2fc_series,
        core_fixed_motifs_df
    )
    if len(log2fc_series_for_flexible_analysis) == len(log2fc_series):
        print("No sequences were filtered out based on fixed-position core motifs (or no core motifs found).")
    else:
        print(f"Filtered log2fc_series for flexible analysis: {len(log2fc_series_for_flexible_analysis)} sequences remaining.")

    if log2fc_series_for_flexible_analysis.empty:
        print("No sequences remaining for flexible motif analysis after filtering.")
        flexible_motifs_results_df = pd.DataFrame(columns=['motif', 'pattern', 'fraction_depleted', 'num_sequences', 'avg_log2fc'])
        return core_fixed_motifs_df, flexible_motifs_results_df

    short_sequences_for_flex_analysis = log2fc_series_for_flexible_analysis.index.tolist()
    sequences_with_context_for_flex = get_lib_seq_context(
        short_sequences_for_flex_analysis,
        left_context_str,
        right_context_str
    )
    fc_values_for_flexible_analysis = log2fc_series_for_flexible_analysis.tolist()
    # NOTE: log2fc_series_for_flexible_analysis is kept alive intentionally —
    # it is passed to update_motif_scores_from_unique_hits below.

    if not isinstance(flexible_motif_patterns, list):
        print("Warning: flexible_motif_patterns should be a list. Wrapping it in a list.")
        flexible_motif_patterns = [flexible_motif_patterns]

    flexible_motif_patterns = sorted(flexible_motif_patterns, key=lambda x: (x[0] + x[2], x[1]))

    print(f"Parallelizing flexible motif analysis across {len(flexible_motif_patterns)} patterns...")
    print(f"Using {cpu_count} processes for parallel analysis.")

    partial_process_pattern = functools.partial(
        process_single_flexible_pattern,
        sequences_with_context_for_flex=sequences_with_context_for_flex,
        fc_values_for_flexible_analysis=fc_values_for_flexible_analysis,
        flexible_motif_log2fc_thr=flexible_motif_log2fc_thr,
        flexible_motif_score_thr=flexible_motif_score_thr,
        flexible_motif_min_support=flexible_motif_min_support
    )

    all_pattern_results = []
    with Pool(processes=cpu_count) as pool:
        for result_df in tqdm(pool.imap_unordered(partial_process_pattern, flexible_motif_patterns),
                              total=len(flexible_motif_patterns),
                              desc="Processing flexible patterns"):
            if not result_df.empty:
                all_pattern_results.append(result_df)

    cumulative_flexible_motifs_df = pd.concat(all_pattern_results, ignore_index=True) if all_pattern_results else pd.DataFrame()

    if not cumulative_flexible_motifs_df.empty:
        print(f"Found {len(cumulative_flexible_motifs_df)} candidate(s) from all patterns. Filtering redundancies...")
        cumulative_flexible_motifs_df = filter_redundant_patterns(
            cumulative_flexible_motifs_df,
            score_margin=core_flexible_motif_score_margin,
        )
        print(f"  -> {len(cumulative_flexible_motifs_df)} core motifs remain after filtering.")

    if cumulative_flexible_motifs_df.empty:
        print(f"\nNo flexible motifs found meeting initial support > {flexible_motif_min_support} and score > {flexible_motif_score_thr} for any provided pattern.")
        flexible_motifs_results_df = pd.DataFrame(columns=['motif', 'pattern', 'fraction_depleted', 'num_sequences', 'avg_log2fc'])
    else:
        n_candidates = len(cumulative_flexible_motifs_df)
        print(f"\nIdentified {n_candidates} raw flexible motifs. Now filtering to core flexible motifs...")

        if n_candidates > max_candidates:
            warnings.warn(
                f"\n{'='*70}\n"
                f"TOO MANY CANDIDATES ({n_candidates} > max_candidates={max_candidates}).\n"
                f"The unique-hit scoring step is O(N²) per sequence and would run\n"
                f"effectively forever with this many candidates.\n\n"
                f"This usually means the thresholds are too permissive for this data.\n"
                f"Suggestions to reduce candidates:\n"
                f"  - Increase flexible_motif_log2fc_thr  (currently {flexible_motif_log2fc_thr}, try e.g. -0.8 or -1.0)\n"
                f"  - Increase flexible_motif_score_thr   (currently {flexible_motif_score_thr}, try e.g. 0.7 or 0.85)\n"
                f"  - Increase flexible_motif_min_support (currently {flexible_motif_min_support}, try e.g. 5 or 10)\n"
                f"  - Increase max_candidates if you know what you are doing.\n"
                f"\nReturning the raw (unfiltered) candidates so you can inspect them.\n"
                f"{'='*70}",
                stacklevel=2
            )
            print(f"\nTop candidates by fraction_depleted:")
            print(cumulative_flexible_motifs_df.sort_values('fraction_depleted', ascending=False).head(20).to_string())
            return core_fixed_motifs_df, cumulative_flexible_motifs_df

        print("\nNow filtering to remove sequences with multiple motifs from the analysis and dropping motifs that no longer pass the thresholds")
        print(f"(Scoring {n_candidates} candidate motifs — this step is O(N²), may take a moment...)")

        flexible_motifs_results_df = update_motif_scores_from_unique_hits(
            cumulative_flexible_motifs_df,
            log2fc_series_for_flexible_analysis,
            left_context_str,
            right_context_str,
            flexible_motif_log2fc_thr=flexible_motif_log2fc_thr,
            flexible_motif_min_support=flexible_motif_min_support,
            flexible_motif_score_thr=flexible_motif_score_thr
        )

        print(f"\n{len(flexible_motifs_results_df)} motifs remaining:")
        if flexible_motifs_results_df.empty:
            print("No core flexible motifs remained after filtering.")
        else:
            for _, row in flexible_motifs_results_df.iterrows():
                print(f"{row['motif']} (from pattern {row['pattern']}), FracDep: {row['fraction_depleted']:.2f}, AvgFC: {row['avg_log2fc']:.2f}, N: {row['num_sequences']}")

    return core_fixed_motifs_df, flexible_motifs_results_df

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()